# Jev-style decision model: Qwen + MLP on (document, hypothesis) pairs

Every class is written as a **hypothesis**. Qwen reads the document together with one hypothesis, and an MLP head outputs one score: the probability that the hypothesis is true. The class with the highest score wins.

Because the model scores hypotheses instead of fixed classes, new yes/no questions and new options work **without retraining**, like Jev (see `decide()` at the end).

**Data:** `data/train.csv`, `data/valid.csv`, `data/test.csv` with columns `text`, `label`, `source` (see the repository README).
**Colab:** open this notebook from GitHub (*File → Open notebook → GitHub*). Then *Runtime → Change runtime type → T4 GPU*, and *Run all*. The notebook downloads `data/` from this repo by itself.
**Locally:** run from the repository folder; needs an NVIDIA GPU.
**Classes:** edit `classes` in the settings cell (label → hypothesis). That is the only place.

In [ ]:
%pip install -q -U "transformers>=5.18" datasets accelerate scikit-learn matplotlib

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

# Data: no data/ folder here (e.g. on Colab) -> download the public repo, which includes it
if not os.path.exists("data"):
    os.system("git clone -q https://github.com/Rozzeo/my_jev.git")
    os.chdir("my_jev")
data_dir = "data"                                     # train.csv, valid.csv, test.csv: text, label, source
weights_path = "qwen_jev_pairs_weights.pt"

# ============ CLASSES: label in the data -> hypothesis (a full statement). The only place to edit ============
classes = {
    "marketing": "This is a marketing document that promotes a product or service.",
    "legal":     "This is a legal document such as a contract, policy or terms.",
    "other":     "This document is neither marketing nor legal.",
}
# =============================================================================================================

# Settings
model_name = "Qwen/Qwen3-0.6B"    # or "Qwen/Qwen3-Embedding-0.6B", "Qwen/Qwen3-1.7B" (all Apache 2.0)
max_doc_tokens = 384              # the document is cut to this, so the hypothesis after it always fits
max_length = 512                  # max tokens per pair
unfreeze_last = 7                 # train the last N transformer layers + the MLP head; the rest stays frozen
use_nli, n_nli = True, 2000       # public NLI pairs teach "the text supports this statement" -> new questions work
seed = 42

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load data: the train / valid / test split from data/ (built by make_dataset.py)
assert os.path.exists(f"{data_dir}/train.csv"), f"Put train.csv, valid.csv and test.csv into {data_dir}"

def load_split(name):
    d = pd.read_csv(f"{data_dir}/{name}.csv").dropna(subset=["text", "label"])
    d["label"] = d["label"].astype(str).str.strip().str.lower()
    return d.reset_index(drop=True)

train_df, valid_df, test_df = load_split("train"), load_split("valid"), load_split("test")
found = set(train_df["label"]) | set(valid_df["label"]) | set(test_df["label"])
assert found <= set(classes), f"labels missing from classes: {found - set(classes)}"

labels = list(classes)                              # class names; position = class id
num_classes = len(labels)
print(f"Number of classes: {num_classes} -> {labels}")
print(f"train {len(train_df)} | valid {len(valid_df)} | test {len(test_df)} documents")
print(train_df["label"].value_counts())

In [ ]:
# Build (document, hypothesis) pairs
tokenizer = AutoTokenizer.from_pretrained(model_name)

def cut_doc(text):
    """Cut the document to max_doc_tokens so truncation never removes the hypothesis at the end."""
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= max_doc_tokens else tokenizer.decode(ids[:max_doc_tokens])

def pair_text(document, hypothesis):
    return f"Document:\n{cut_doc(document)}\n\nHypothesis: {hypothesis}\nIs the hypothesis true for this document? Answer:"

def make_pairs(documents):
    """Each document -> one pair per class, in classes order."""
    return [pair_text(doc, hyp) for doc in documents for hyp in classes.values()]

def pair_targets(doc_labels):
    """1.0 for the pair with the document's true class, 0.0 for the others."""
    return [float(c == label) for label in doc_labels for c in classes]

# Public NLI pairs (MNLI): premise -> hypothesis; label 0 = entailment = "true"
nli_texts, nli_targets = [], []
if use_nli:
    nli = load_dataset("nyu-mll/multi_nli", split=f"train[:{n_nli}]")
    nli_texts = [pair_text(p, h) for p, h in zip(nli["premise"], nli["hypothesis"])]
    nli_targets = [float(l == 0) for l in nli["label"]]

train_texts = make_pairs(train_df["text"]) + nli_texts
train_targets = pair_targets(train_df["label"]) + nli_targets
ds = DatasetDict({
    "train": Dataset.from_dict({"text": train_texts, "label": train_targets}),
    "validation": Dataset.from_dict({"text": make_pairs(valid_df["text"]), "label": pair_targets(valid_df["label"])}),
    "test": Dataset.from_dict({"text": make_pairs(test_df["text"]), "label": pair_targets(test_df["label"])}),
})
print(ds)
print("\nExample pair:\n" + ds["validation"][0]["text"])

In [ ]:
# ================================ HEAD: MLP ================================
# Reads Qwen's vector of a (document, hypothesis) pair and outputs ONE score: "the hypothesis is true".
# One score per pair -> more classes or new questions need no change here.
# Lots of data or many classes: raise the hidden size 256 -> 512.
# ===========================================================================
def mlp_head(hidden_size):
    return nn.Sequential(
        nn.LayerNorm(hidden_size),                  # LLM vectors have a few huge-magnitude dimensions
        nn.Linear(hidden_size, 256), nn.GELU(), nn.Dropout(0.1),
        nn.Linear(256, 1),
    )

def build_model():
    # A warning that `score` weights are newly initialized is expected: that head is replaced right below
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1, dtype=torch.float32)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model.score = mlp_head(model.config.hidden_size)    # replace the default linear head with the MLP
    return model

model = build_model()

# Transfer learning: freeze everything, then unfreeze the last N layers, the final norm and the MLP head
for param in model.parameters():
    param.requires_grad = False
for module in [*model.model.layers[-unfreeze_last:], model.model.norm, model.score]:
    for param in module.parameters():
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable / 1e6:.1f}M of {model.num_parameters() / 1e6:.0f}M")

In [ ]:
# Tokenize
def preprocess_function(batch):
    inputs = tokenizer(batch["text"], truncation=True, max_length=max_length)
    inputs["labels"] = batch["label"]
    return inputs

tokenized_ds = ds.map(preprocess_function, batched=True, remove_columns=ds["train"].column_names)

# Loss: binary cross-entropy on each pair. Only ~1 in K pairs is true, so pos_weight balances them
# (then 0.5 stays a fair yes/no threshold)
pos_rate = np.mean(train_targets)
pos_weight = torch.tensor((1 - pos_rate) / pos_rate, dtype=torch.float32)

def pair_loss(outputs, labels, num_items_in_batch=None):
    logits = outputs.logits.view(-1).float()
    return F.binary_cross_entropy_with_logits(logits, labels.float(), pos_weight=pos_weight.to(logits.device))

# Metrics: K pair scores per document -> the class with the highest score
def compute_metrics(eval_pred):
    scores = eval_pred.predictions.reshape(-1, num_classes)            # [documents, classes]
    y_true = eval_pred.label_ids.reshape(-1, num_classes).argmax(1)    # the pair with target 1 = true class
    y_pred = scores.argmax(1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),         # every class counts equally
    }

In [ ]:
# Two learning rates: the new MLP head learns fast, the pretrained Qwen layers gently
model.to(device)
optimizer = torch.optim.AdamW([
    {"params": model.score.parameters(), "lr": 1e-3},
    {"params": [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("score")], "lr": 2e-5},
], weight_decay=0.01)

# Set training arguments
training_args = TrainingArguments(
    output_dir="checkpoints",                # local and temporary
    per_device_train_batch_size=16,          # out of GPU memory? use 8
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    warmup_steps=0.1,                        # a float = share of steps: first 10% warm up the learning rate
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",        # keep the epoch with the best validation macro-F1
    greater_is_better=True,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),          # mixed precision on the GPU
    report_to="none",
    seed=seed,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_loss_func=pair_loss,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),            # None -> Trainer adds the warmup + linear decay schedule
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],   # stop when an epoch brings no improvement
)

# Train the model
trainer.train()

In [ ]:
# Training history: loss and validation metrics per epoch
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "axes.axisbelow": True, "lines.linewidth": 2})
history = pd.DataFrame(trainer.state.log_history)
train_log = history.dropna(subset=["loss"])
eval_log = history.dropna(subset=["eval_loss"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(train_log["epoch"], train_log["loss"], color="#2a78d6", label="train")
ax1.plot(eval_log["epoch"], eval_log["eval_loss"], "o-", color="#eb6834", label="validation")
ax1.set(title="Loss (per pair)", xlabel="epoch")
ax1.legend()
ax2.plot(eval_log["epoch"], eval_log["eval_f1_macro"], "o-", color="#2a78d6", label="macro-F1")
ax2.plot(eval_log["epoch"], eval_log["eval_accuracy"], "o-", color="#eb6834", label="accuracy")
ax2.set(title="Validation metrics (per document)", xlabel="epoch", ylim=(0, 1.05), xticks=eval_log["epoch"])
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on the test set (never used for training or picking the best epoch)
test_output = trainer.predict(tokenized_ds["test"])
test_pred = test_output.predictions.reshape(-1, num_classes).argmax(1)
test_true = test_df["label"].map(labels.index).values
print(f"Test accuracy: {test_output.metrics['test_accuracy']:.4f} | test macro-F1: {test_output.metrics['test_f1_macro']:.4f}")
print(classification_report(test_true, test_pred, labels=range(num_classes), target_names=labels, digits=3, zero_division=0))

# Confusion matrix
fig, ax = plt.subplots(figsize=(num_classes + 3, num_classes + 2))
ConfusionMatrixDisplay.from_predictions(test_true, test_pred, labels=range(num_classes), display_labels=labels,
                                        cmap="Blues", colorbar=False, xticks_rotation=45, ax=ax)
ax.set_title("Confusion matrix (test)")
ax.grid(False)
plt.tight_layout()
plt.show()

## `decide()`: Jev-like questions

The request mirrors Jev: `state` (the document) + typed `questions`. Hypotheses can be new, with no retraining.

- `choice`: `criteria` maps option → hypothesis; the highest probability wins.
- `noul`: `instructions` is one statement; returns yes/no plus probability.
- `score`: `criteria` is a list of levels; returns the index of the most likely level.

Write hypotheses as **full statements**. Questions far from the training data are only as reliable as the NLI mix allows, so check them on a few real documents.

In [ ]:
def pair_probs(pairs):
    """Pair texts -> probability that each hypothesis is true (same order as the input)."""
    new_ds = Dataset.from_dict({"text": pairs}).map(
        lambda b: tokenizer(b["text"], truncation=True, max_length=max_length), batched=True, remove_columns=["text"])
    return torch.sigmoid(torch.tensor(trainer.predict(new_ds).predictions).view(-1)).tolist()

def decide(state, questions):
    answers = {}
    for name, q in questions.items():
        if q["type"] == "choice":            # criteria = {option: hypothesis} -> highest probability wins
            options = list(q["criteria"])
            p = pair_probs([pair_text(state, h) for h in q["criteria"].values()])
            best = int(np.argmax(p))
            answers[name] = {"choice": options[best], "confidence": round(p[best], 4),
                             "scores": dict(zip(options, np.round(p, 4).tolist()))}
        elif q["type"] == "noul":            # instructions = one statement -> yes / no
            p = pair_probs([pair_text(state, q["instructions"])])[0]
            answers[name] = {"value": p >= 0.5, "probability": round(p, 4)}
        elif q["type"] == "score":           # criteria = [level statements] -> index of the most likely level
            p = pair_probs([pair_text(state, h) for h in q["criteria"]])
            best = int(np.argmax(p))
            answers[name] = {"score": best, "level": q["criteria"][best], "scores": np.round(p, 4).tolist()}
        else:
            raise ValueError(f"Unknown question type: {q['type']}")
    return {"answers": answers}

# Example: the trained classes + a brand-new yes/no question the model never saw in training
document = test_df["text"].iloc[0]
print(document[:300], "...\n")
print(json.dumps(decide(document, {
    "class": {"type": "choice", "criteria": classes},
    "mentions_money": {"type": "noul", "instructions": "The document mentions money, prices or payments."},
}), indent=2))

In [ ]:
# Batch mode for iConduct: best class + confidence for many documents; low confidence -> manual review
def classify_batch(texts, threshold=0.6):
    p = torch.tensor(pair_probs(make_pairs(texts))).view(-1, num_classes)    # [documents, classes]
    confidence, best = p.max(dim=1)
    return pd.DataFrame({
        "pred_label": [labels[i] for i in best.tolist()],
        "confidence": confidence.double().numpy().round(4),
        "needs_review": (confidence < threshold).numpy(),
    })

predictions = pd.concat([test_df[["text", "label"]].reset_index(drop=True), classify_batch(test_df["text"])], axis=1)
predictions.to_csv("predictions.csv", index=False)
print("needs review:", int(predictions["needs_review"].sum()), "of", len(predictions))

# Save only what was trained (last layers + MLP head): ~0.45 GB instead of 2.4 GB for the whole model.
# Load later: model = build_model(); model.load_state_dict(torch.load(weights_path), strict=False)
torch.save({n: p.detach().cpu() for n, p in trainer.model.named_parameters() if p.requires_grad}, weights_path)
predictions.head(10)